In [7]:
#!pip install skrebate scikit-learn pandas numpy matplotlib seaborn

$\textbf{Step 0: Importing all necessary libraries}$

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import pickle
warnings.filterwarnings('ignore')

#Importing The ReliefF Library to Implement for Phase 2
from skrebate import ReliefF

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

#Importing Libraries for the 5 Classifiers that will be used
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

$\textbf{Step 1: Loading Datasets and Setting the Parameters for Models and}$
$\textbf{ReliefF Algorithm}$

In [9]:
DATASETS = {
    'Dataset1' : {'train': 'Datasets/1/train.csv','test': None},
    'Dataset2' : {'train': 'Datasets/2/train.csv','test': None},
    'Dataset3' : {'train': 'Datasets/3/train.csv','test': None},
    'Dataset4' : {'train': 'Datasets/4/train.csv','test': None},
    'Dataset5' : {'train': 'Datasets/5/train.csv','test': None},
    'Dataset6' : {'train': 'Datasets/6/train.csv','test': None},
    'Dataset7' : {'train': 'Datasets/7/train.csv','test': None},
    'Dataset8' : {'train': 'Datasets/8/train.csv','test': None},
}

LABEL_COL = 'Label'
N_FOLDS = 10
INNER_FOLDS = 3
RANDOM_STATE = 42

# ReliefF Settings
# k = number of top features to keep after ranking
# Try a few values; the best k is selected by inner CV
K_CANDIDATES = [5, 10, 20]  # adjust based on total feature count
RELIEFF_N_NEIGHBORS = 10          # ReliefF neighbourhood size


$\textbf{Step 2: Classifier Definition with Hyperparameter Grids}$

In [10]:
CLASSIFIERS = {
    'SVM': (
            CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, max_iter=2000)),
            {'classifier__estimator__C': [1, 10]}
            ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,)],
            'classifier__alpha':              [1e-4, 1e-3],
        }
    ),
}

$\textbf{Step 3: Helper Functions}$

In [11]:
def load_dataset(path, label_col=LABEL_COL):
    df = pd.read_csv(path)
    X = df.drop(columns=[label_col]).values.astype(np.float64)
    y_raw = df[label_col].values
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    print(f'  Loaded: {X.shape[0]} samples, {X.shape[1]} features, '
          f'{len(np.unique(y))} classes')
    return X, y, le

def apply_relieff(X_train, y_train, X_test, k, n_neighbors=RELIEFF_N_NEIGHBORS, max_samples=500):
    if X_train.shape[0] > max_samples:
        idx = np.random.RandomState(RANDOM_STATE).choice(
            X_train.shape[0], max_samples, replace=False)
        X_relief = X_train[idx]
        y_relief = y_train[idx]
    else:
        X_relief = X_train
        y_relief = y_train

    imputer = SimpleImputer(strategy='mean')
    X_relief_imp = imputer.fit_transform(X_relief)

    selector = ReliefF(n_features_to_select=k, n_neighbors=n_neighbors)
    selector.fit(X_relief_imp, y_relief)
    top_idx = np.argsort(selector.feature_importances_)[::-1][:k]
    return X_train[:, top_idx], X_test[:, top_idx], selector.feature_importances_, top_idx

def build_pipeline(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', clf),
    ])

def evaluate_classifier(clf, param_grid, X_train, y_train, X_test, y_test):
    pipe = build_pipeline(clf)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True,
                                random_state = RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division = 0)
    best_params = gs.best_params_
    return acc, f1, best_params


$\textbf{Step 4: Main Experiment Loop}$

In [12]:
all_results = {}
relieff_scores = {}

outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for ds_name, paths in DATASETS.items():
    print(f'\n=== Dataset: {ds_name} ===')
    X, y, le = load_dataset(paths['train'])

    all_results[ds_name] = {
        'baseline': {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
        'relieff':  {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
    }
    fold_feature_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f'  Fold {fold_idx + 1}/{N_FOLDS}', end=' ... ')
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Phase 1: baseline (full features)
        for clf_name, (clf, grid) in CLASSIFIERS.items():
            t0 = time.time()
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train, y_train, X_test, y_test)
            all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
            all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
            all_results[ds_name]['baseline'][clf_name]['params'].append(params)
            print(f"    {clf_name}: {time.time()-t0:.1f}s")

        # Phase 2: ReliefF
        print("    Starting ReliefF...", end=" ", flush=True)
        t0 = time.time()
        _, _, importances, sorted_idx = apply_relieff(
            X_train, y_train, X_test, k=1)
        print(f"done ({time.time()-t0:.1f}s)", flush=True)

        best_k, best_k_score = K_CANDIDATES[0], -np.inf
        for k_candidate in K_CANDIDATES:
            if k_candidate > X_train.shape[1]:
                continue
            top_idx = np.argsort(importances)[::-1][:k_candidate]
            Xtr_k, Xte_k = X_train[:, top_idx], X_test[:, top_idx]
            inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
            scores = []
            for itr, ite in inner.split(Xtr_k, y_train):
                rf = RandomForestClassifier(n_estimators=50,
                                            random_state=RANDOM_STATE, n_jobs=-1)
                rf.fit(Xtr_k[itr], y_train[itr])
                scores.append(f1_score(y_train[ite],
                                       rf.predict(Xtr_k[ite]),
                                       average='macro', zero_division=0))
            if np.mean(scores) > best_k_score:
                best_k_score = np.mean(scores)
                best_k = k_candidate

        print(f"    best_k selected: {best_k}", flush=True)

        top_idx_final = np.argsort(importances)[::-1][:best_k]
        X_train_r = X_train[:, top_idx_final]
        X_test_r  = X_test[:, top_idx_final]
        fold_feature_scores.append(importances)

        for clf_name, (clf, grid) in CLASSIFIERS.items():
            t0 = time.time()
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train_r, y_train, X_test_r, y_test)
            all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
            all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
            all_results[ds_name]['relieff'][clf_name]['params'].append(params)
            print(f"    {clf_name}: {time.time()-t0:.1f}s")

        print('done')

    relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)

# Save checkpoint
with open('results_checkpoint.pkl', 'wb') as f:
    pickle.dump({'all_results': all_results, 'relieff_scores': relieff_scores}, f)
print('Checkpoint saved.')


=== Dataset: Dataset1 ===
  Loaded: 9583 samples, 19 features, 4 classes
  Fold 1/10 ...     SVM: 13.4s
    kNN: 1.0s
    Decision Tree: 0.6s
    Random Forest: 5.8s
    MLP: 7.8s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.7s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 4.0s
    MLP: 1.8s
done
  Fold 2/10 ...     SVM: 1.3s
    kNN: 0.9s
    Decision Tree: 0.5s
    Random Forest: 6.9s
    MLP: 6.6s
    Starting ReliefF... done (0.9s)
    best_k selected: 5
    SVM: 0.4s
    kNN: 0.3s
    Decision Tree: 0.2s
    Random Forest: 4.1s
    MLP: 2.3s
done
  Fold 3/10 ...     SVM: 1.6s
    kNN: 1.0s
    Decision Tree: 0.6s
    Random Forest: 6.9s
    MLP: 7.0s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.5s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 3.5s
    MLP: 1.4s
done
  Fold 4/10 ...     SVM: 1.2s
    kNN: 1.1s
    Decision Tree: 0.6s
    Random Forest: 6.5s
    MLP: 7.5s
    Starting ReliefF... done (0.7s)
   

In [6]:
import pickle
import pandas as pd
import numpy as np

# Load pickle
with open('results_checkpoint.pkl', 'rb') as f:
    data = pickle.load(f)

all_results = data['all_results']

model_map = {
    'SVM': 'SVM',
    'kNN': 'KNN',
    'Decision Tree': 'DT',
    'Random Forest': 'RF',
    'MLP': 'MLP'
}

# Columns
columns = []
metric_cols = []
for m in model_map.values():
    columns += [f"{m}-Accuracy", f"{m}-F1"]
    metric_cols += [f"{m}-Accuracy", f"{m}-F1"]
columns += [f"{m} Parameters" for m in model_map.values()]


def build_sheet(phase):
    all_rows = []

    datasets = sorted(all_results.keys())

    for i, ds in enumerate(datasets, start=1):
        rows = []

        data_block = all_results[ds][phase]

        # Header
        rows.append([f"Data {i}"] + [""] * len(columns))
        rows.append([""] + columns)

        n_folds = len(next(iter(data_block.values()))['acc'])

        fold_data = []

        for fold in range(n_folds):
            row = [f"Fold {fold+1}"]
            numeric_row = []

            # Metrics
            for model in model_map:
                acc = data_block[model]['acc'][fold]
                f1 = data_block[model]['f1'][fold]

                row.append(acc)
                row.append(f1)

                numeric_row.extend([acc, f1])

            # Params
            for model in model_map:
                row.append(str(data_block[model]['params'][fold]))

            rows.append(row)
            fold_data.append(numeric_row)

        # Convert to array for stats
        fold_array = np.array(fold_data)

        # Mean row
        mean_vals = np.mean(fold_array, axis=0)
        mean_row = ["Mean"] + list(mean_vals) + [""] * len(model_map)

        # Std row
        std_vals = np.std(fold_array, axis=0)
        std_row = ["Std"] + list(std_vals) + [""] * len(model_map)

        rows.append(mean_row)
        rows.append(std_row)

        # Spacer
        rows.append([""] * (len(columns) + 1))

        all_rows.extend(rows)

    return pd.DataFrame(all_rows)


# Build sheets
df_baseline = build_sheet('baseline')
df_relieff = build_sheet('relieff')

# Save Excel
with pd.ExcelWriter('formatted_results.xlsx', engine='openpyxl') as writer:
    df_baseline.to_excel(writer, sheet_name='baseline', index=False, header=False)
    df_relieff.to_excel(writer, sheet_name='relieff', index=False, header=False)

print("✅ Excel file created with Mean + Std")

✅ Excel file created with Mean + Std
